# 01 · Download and discover IMPACT (M1)
Downloads only what V1 needs and inventories it:
1. the annotations bundle (8 MiB, SHA256-checked) and the 2.8 GB quick-start sample (to local scratch, for adapter tests);
2. the 45 Model A Reassembly **front-view videos** and **VideoMAEv2 features**, pulled out of the big remote zips with
   HTTP range requests (no full 13–29 GiB / 21.6 GiB downloads; each file is CRC-checked).

**Paste back** the outputs marked 📋 so they can go into `docs/DATA_NOTES.md`.

## Setup
Mounts Drive, clones or updates the repo, installs it and sets the `SOPVERIFY_*` paths. Safe to re-run after a runtime reset.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
%%bash
set -e
REPO=/content/SOP-Verification
BRANCH=claude/nifty-mccarthy-5suut3   # switch to main once the branch is merged
if [ -d $REPO/.git ]; then
  git -C $REPO fetch -q origin $BRANCH && git -C $REPO checkout -q $BRANCH && git -C $REPO pull -q --ff-only origin $BRANCH
else
  git clone -q -b $BRANCH https://github.com/RohanSada/SOP-Verification.git $REPO
fi
pip install -q -e "$REPO[app,dev]"
git -C $REPO log --oneline -1

In [ ]:
import os, sys
sys.path.insert(0, "/content/SOP-Verification/src")   # editable install without a runtime restart
os.chdir("/content/SOP-Verification")
from sopverify.colab import setup_env, show
ROOT = "/content/drive/MyDrive/sopverify"
ENV = setup_env(ROOT)
try:  # optional Hugging Face token (Colab secret named HF_TOKEN), in case the dataset is gated
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass
ENV

### 1. Annotations bundle

In [ ]:
!sopverify data download annotations
!ls $SOPVERIFY_DATA_ROOT; ls $SOPVERIFY_DATA_ROOT/annotations | head

### 2. Inventory 📋

In [ ]:
!sopverify data discover --out "$SOPVERIFY_EXPORTS/inventory_summary.md"

### 3. Metadata (expert vs novice) 📋
The GitHub copy of the annotations has no expertise labels; they should be in `metadata/`.

In [ ]:
from sopverify.data.impact.catalog import Catalog
cat = Catalog.from_paths(os.environ["SOPVERIFY_DATA_ROOT"], None)
print("metadata files:", len(cat.index.metadata_files))
for p in cat.index.metadata_files[:20]:
    print("\n==", p); print(open(p, errors="ignore").read()[:1500])
print("\nexpertise parsed:", cat.expertise)

### 4. Pool summary 📋

In [ ]:
!sopverify data summarize --out "$SOPVERIFY_EXPORTS/pool_summary.csv"

### 5. Quick-start sample → adapter tests on real files

In [ ]:
!sopverify data download sample
import glob
cands = glob.glob("/content/scratch/impact_sample/**/annotations", recursive=True) or ["/content/scratch/impact_sample"]
os.environ["SOPVERIFY_TEST_ANNOTATIONS"] = cands[0]; print(cands[0])
!python -m pytest -q -m data 2>&1 | tail -5

### 6. What is inside the big bundles? (zip directory only) 📋

In [ ]:
from sopverify.data.impact.download import remote_listing
pool = cat.execution_ids()
for key in ["videos:front", "features:VideoMAEv2"]:
    lst = remote_listing(key)
    want = [(n, s) for n, s in lst if any(f"{e}_front" in n for e in pool)]
    print(key, "members:", len(lst), "| pool members:", len(want), "| pool size: %.1f GB" % (sum(s for _, s in want) / 1e9))
    print("  e.g.", want[:2])

### 7. Fetch pool videos and VideoMAEv2 features (resumable)

In [ ]:
!sopverify data download videos
!sopverify data download features --model VideoMAEv2

### 8. Verify resolution, fps and feature layout 📋

In [ ]:
import numpy as np, pandas as pd
from sopverify.video.decode import probe
from sopverify.features.shipped import load_feature_array
cat = Catalog.from_paths(os.environ["SOPVERIFY_DATA_ROOT"], None)
rows = []
for e in cat.execution_ids():
    ref = cat.ref(e); row = {"execution": e, "asr_frames": ref.n_frames}
    if ref.video_path:
        i = probe(ref.video_path); row.update(w=i.width, h=i.height, fps=round(i.fps, 3), video_frames=i.n_frames)
    if "VideoMAEv2" in ref.shipped_features:
        raw = np.load(ref.shipped_features["VideoMAEv2"], mmap_mode="r"); row.update(feat_shape=str(raw.shape))
        row["feat_T"] = load_feature_array(ref.shipped_features["VideoMAEv2"], 1408).shape[0]
    rows.append(row)
df = pd.DataFrame(rows); print(df.to_string())
print(df[["w", "h", "fps"]].value_counts() if "w" in df else "no videos")
if "feat_T" in df: print("feat_T - asr_frames:", (df.feat_T - df.asr_frames).describe())

In [ ]:
!sopverify data summarize | tail -3
!du -sh "$SOPVERIFY_DATA_ROOT"